# HealthYarn Batch TTS Generator (YarnGPT2b on Google Colab GPU)

This notebook runs `backend/scripts/generate_audio.py` on a Colab GPU using `saheedniyi/YarnGPT2b` (`AudioTokenizerV2`) to synthesize approved Yorùbá, Igbo, and Nigerian English health answers into Opus `.ogg` clips (24–32 kbps mono, < 150 KB each).

**Hard rule**: YarnGPT is used offline for text-to-speech only, never at runtime and never for ASR.

In [ ]:
# 1. Mount Google Drive, clone saheedniyi02/yarngpt (v2 AudioTokenizerV2), and install dependencies
from google.colab import drive
drive.mount('/content/drive')

!apt-get -y install ffmpeg
!rm -rf /content/yarngpt_repo
!git clone https://github.com/saheedniyi02/yarngpt.git /content/yarngpt_repo
# Ensure we do NOT install python-wrapper (which is v1 English-only and shadows AudioTokenizerV2)
!pip uninstall -y yarngpt || true
!pip install -q outetts==0.2.3 uroman torchaudio transformers accelerate sqlmodel pydantic-settings numpy bcrypt rapidfuzz slowapi PyJWT spitch "protobuf==4.25.3"

In [ ]:
# 2. Download WavTokenizer configuration & weights directly from Hugging Face
!wget -q --show-progress -O /content/wavtokenizer_mediumdata_frame75_3s_nq1_code4096_dim512_kmeans200_attn.yaml https://huggingface.co/novateur/WavTokenizer-medium-speech-75token/resolve/main/wavtokenizer_mediumdata_frame75_3s_nq1_code4096_dim512_kmeans200_attn.yaml
!wget -q --show-progress -O /content/wavtokenizer_large_speech_320_24k.ckpt https://huggingface.co/novateur/WavTokenizer-large-speech-75token/resolve/main/wavtokenizer_large_speech_320_24k.ckpt

In [ ]:
# 3. Point to your HealthYarn repository folder in Drive and generate YarnGPT2b Opus clips
import os, sys
REPO_DIR = '/content/drive/MyDrive/HealthYarn'
os.chdir(f'{REPO_DIR}/backend')

# Seed database if needed, then run multilingual AudioTokenizerV2 generation (2 takes per answer)
!PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python python -m scripts.seed --demo
!PYTHONPATH=/content/yarngpt_repo:. PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python python -m scripts.generate_audio --takes 2

In [ ]:
# 4. Zip generated .ogg files + DB for download back into your local repo
!cd /content/drive/MyDrive/HealthYarn/backend/static && zip -r /content/healthyarn_audio_ogg.zip audio/
!cp /content/drive/MyDrive/HealthYarn/backend/healthyarn.db /content/healthyarn.db
from google.colab import files
files.download('/content/healthyarn_audio_ogg.zip')